# 05 — Platform sizing

**Code under test:** `fleximorpv2/models/platform.py`

**Purpose:** Check platform area, load limits, and depth-driven platform type.

Run cells **top to bottom**. Each markdown cell explains the **next code cell** — what it does, what to inspect in the output, and what counts as a pass.

Track overall audit progress in Obsidian (`FlexiMORP Calculation Audit.md`). These notebooks are the lab workbook, not the checklist.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Step 1 — Design platform from sample variables

**Run the next cell** with a fixed design dict.

**Pass if:** `design_platform` returns specs without error; footprint scales with area; document that `load_utilization=0.8` in platform.py is a placeholder, not measured load.

In [ ]:
from fleximorpv2.config import load_config
from fleximorpv2.models.platform import PlatformModel

config = load_config("blyth")
platform = PlatformModel(config)
design_vars = {
    "wind_capacity": 50.0,
    "solar_capacity": 10.0,
    "wave_capacity": 5.0,
    "platform_area": 50_000,
    "water_depth": 35,
    "distance_to_shore": 5,
}
tech_perf = {"total_load_requirement": 8000, "total_space_requirement": 40_000}
specs = platform.design_platform(design_vars, tech_perf)
print(specs)